# 🤖📰 Treinar a IA de Notícias — SmartTrader

Este notebook treina/usa a **IA que lê notícias e decide direção** (comprar/vender),
que é o coração da sua ideia. Roda no **Google Colab** (use GPU: *Ambiente de execução → Alterar tipo → GPU*).

### ⚠️ Honestidade primeiro (leia)
- O **FinBERT** mede o **sentimento** de um texto financeiro (positivo/negativo/neutro). Ele **já vem treinado**.
- 'Treinar' aqui = **especializar (fine-tune)** o FinBERT num dataset financeiro real (Financial PhraseBank), usando a GPU.
- Sentimento **não é direção** de um ativo. Quem transforma 'petróleo subindo' em 'USDCAD cai' é a camada **macro→direção** (Passo 4).
- Nada disso garante lucro. É um **filtro de qualidade** — validar antes de dinheiro real.


## Passo 1 — Instalar dependências


In [ ]:
!pip install -q transformers datasets torch scikit-learn yfinance pandas
import torch
print('GPU disponível:', torch.cuda.is_available())


## Passo 2 — Usar o FinBERT pronto (já funciona, sem treinar)
Carrega o FinBERT pré-treinado e mede o sentimento de manchetes.


In [ ]:
from transformers import pipeline

finbert = pipeline('text-classification', model='ProsusAI/finbert',
                   device=0 if torch.cuda.is_available() else -1)

manchetes = [
    'Oil prices spike after OPEC supply cut amid Middle East war',
    'Tech stocks rally as inflation cools and growth beats forecasts',
    'Markets plunge as recession fears and conflict escalate',
]
for m in manchetes:
    r = finbert(m)[0]
    print(f"{r['label']:9s} ({r['score']:.2f})  <-  {m}")


## Passo 3 — (GPU) Fine-tunar o FinBERT no Financial PhraseBank
Especializa o modelo num dataset rotulado por especialistas de finanças.
Pode levar alguns minutos na GPU. (Opcional — pule se só quiser usar o Passo 2.)


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np

# Dataset financeiro real (frases rotuladas: 0=negativo,1=neutro,2=positivo)
ds = load_dataset('financial_phrasebank', 'sentences_50agree')['train']
ds = ds.train_test_split(test_size=0.2, seed=42)

MODEL = 'ProsusAI/finbert'
tok = AutoTokenizer.from_pretrained(MODEL)
def prep(b): return tok(b['sentence'], truncation=True, padding='max_length', max_length=128)
ds = ds.map(prep, batched=True)
ds = ds.rename_column('label', 'labels')
ds.set_format('torch', columns=['input_ids','attention_mask','labels'])

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': (preds == p.label_ids).mean()}

args = TrainingArguments(output_dir='/content/finbert_ft', num_train_epochs=1,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy='epoch', logging_steps=50, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=ds['train'],
    eval_dataset=ds['test'], compute_metrics=metrics)
trainer.train()
print('Avaliação:', trainer.evaluate())
trainer.save_model('/content/finbert_ft'); tok.save_pretrained('/content/finbert_ft')
print('Modelo salvo em /content/finbert_ft')


## Passo 4 — De SENTIMENTO para DIREÇÃO (a sua ideia)
Aqui a notícia vira **comprar/vender** por ativo, tratando **forças conflitantes**
(ex.: crise no petróleo fortalece o CAD, mas o safe-haven fortalece o USD).
Esta é a versão resumida do `news_mapper.py` do projeto — transparente e auditável.


In [ ]:
# Temas -> palavras-chave
THEME_KW = {
  'oil': ['oil','crude','opec','brent','wti','petroleum','petroleo'],
  'risk_off': ['war','crisis','conflict','recession','crash','plunge'],
  'risk_on': ['rally','optimism','growth beats','soar'],
}
# Exposição: coef>0 => tema forte empurra o ativo PRA CIMA; coef<0 => pra baixo
EXPO = {
  'USDCAD': {'oil': -0.7, 'risk_off': +0.6},   # petroleo+ => CAD forte => USDCAD cai; risk_off => USD safe-haven
  'XAUUSD': {'risk_off': +0.8, 'oil': +0.2},    # ouro sobe no medo
  'USOIL':  {'oil': +1.0},
  'SPX':    {'risk_off': -0.8, 'risk_on': +0.7},
}

def detectar_temas(textos):
    t = ' '.join(textos).lower()
    temas = {}
    for tema, kws in THEME_KW.items():
        hits = sum(t.count(k) for k in kws)
        if hits: temas[tema] = min(1.0, hits/(hits+1))  # força 0..1
    return temas

def direcao(simbolo, temas):
    net = sum(EXPO.get(simbolo,{}).get(tema,0.0)*forca for tema,forca in temas.items())
    bias = 1 if net > 0.15 else (-1 if net < -0.15 else 0)
    conf = min(1.0, abs(net))
    return bias, conf, net

manchetes = ['Oil prices spike after OPEC supply cut amid Middle East war']
# (opcional) usa o FinBERT pra medir o tom geral da notícia:
tom = finbert(manchetes[0])[0]
print('Sentimento FinBERT:', tom['label'], f"({tom['score']:.2f})")

temas = detectar_temas(manchetes)
print('Temas:', temas)
for s in ['USOIL','XAUUSD','USDCAD','SPX']:
    b, c, net = direcao(s, temas)
    lado = {1:'COMPRA',-1:'VENDA',0:'NEUTRO'}[b]
    print(f'  {s}: {lado}  (conf {c:.2f}, net {net:+.2f})')


## Passo 5 — Plugar no bot (depois)
1. Baixe a pasta `/content/finbert_ft` (modelo fine-tunado).
2. No projeto, implemente `NewsBiasEngine._score_sentiment` carregando esse modelo
   e `_interpret_macro` usando a lógica do Passo 4 (ou o `news_mapper.py`).
3. Ligue `USE_AI=true` no `.env` e rode na conta DEMO primeiro.

> 🎯 Resumo honesto: o FinBERT te dá **sentimento**; a camada macro te dá **direção**.
> Juntos viram o viés que a estratégia técnica confirma (Modo A). Valide em demo
> antes de qualquer real — sentimento de notícia **não é bola de cristal**.
